# 04 Model 2: RNN + Classifier
Training of architecture 2: the same frozen word embeddings as in model 1, a 2-layer RNN (LSTM or GRU) and a 2-layer classifier (ReLU non-linearity). The RNN and the classifier are trained. Experiments are tracked with Weights & Biases.

Load the word numbers, the labels and the embedding matrix that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only used once at the end, in `05_evaluation.ipynb`.

In [2]:
import numpy as np
import torch

data = np.load("features/model2.npz")
# word numbers of O1, O2, h1, h2 for every example, and the labels (0 = h1, 1 = h2)
X_train = torch.tensor(data["X_train"])
y_train = torch.tensor(data["y_train"])
X_valid = torch.tensor(data["X_valid"])
y_valid = torch.tensor(data["y_valid"])
# GloVe vectors: row i = vector of the word with number i (row 0 = padding)
embedding_matrix = torch.tensor(data["embedding_matrix"])
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)
print(embedding_matrix.shape, embedding_matrix.dtype)

torch.Size([16801, 4, 34]) torch.int64
torch.Size([16801]) torch.int64
torch.Size([15465, 300]) torch.float32


## Classifier
The same 2-layer classifier as for model 1 (`03_model1_embeddings.ipynb`, from the course exercise "PyTorch" (Week 1)): linear layer, ReLU, linear layer. In model 2, its input are the final hidden states of the RNN instead of the averaged word vectors.

In [3]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

## Model 2: RNN + classifier
The same LSTM (or GRU) reads each of the four texts (O1, O2, h1, h2) word by word. The final hidden state of its last layer, h_n, summarizes the text (course slides "Advanced RNNs", slide 16). The four final hidden states are put side by side, like the four sentence vectors in model 1, and the classifier makes the decision. The embedding layer contains the frozen GloVe vectors (`freeze=True`), and number 0 is the padding (`padding_idx=0`). With `batch_first=True` the input has the shape [batch size, sequence length, embedding dim], as on slide 14 of the course slides "Preliminaries".

In [4]:
class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # classifier on the 4 final hidden states side by side (O1, O2, h1, h2)
        self.classifier = Classifier(4 * hidden_dim, 2, inner_dim=inner_dim)

# same start weights every time (as in model 1)
torch.manual_seed(42)
model = RNNClassifier(embedding_matrix)
print(model)

RNNClassifier(
  (embedding): Embedding(15465, 300, padding_idx=0)
  (rnn): LSTM(300, 128, num_layers=2, batch_first=True)
  (classifier): Classifier(
    (linear1): Linear(in_features=512, out_features=100, bias=True)
    (relu): ReLU()
    (linear2): Linear(in_features=100, out_features=2, bias=True)
  )
)


## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier` (same as in `03_model1_embeddings.ipynb`), based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Model 2: 2-layer RNN (LSTM or GRU, PyTorch implementations) with the frozen word embeddings of model 1 and a 2-layer classifier with ReLU, as required in the Project 1 description; `nn.Embedding.from_pretrained` with `freeze=True` and `padding_idx=0` from the PyTorch documentation `torch.nn.Embedding`, https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html; `nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers)` as in the course exercise "Sequence-to-sequence RNN" (Week 4); `nn.LSTM`, `nn.GRU` and `dropout` (between the RNN layers) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html and https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html; `batch_first=True` for the input shape [batch size, sequence length, embedding dim] from the course slides "Preliminaries" (Week 1), slide 14; final hidden states of the texts as input of the classifier from the course slides "Advanced RNNs" (Week 4), slides 16-17.
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.
